# `SummarizationMiddleware`

Middleware that automatically summarizes older conversation messages when configured context thresholds are reached.

It replaces the summarized portion of the conversation with a generated summary while preserving recent messages. Cutoff selection avoids separating an AI tool call from its corresponding tool-result messages.

- Bases: `AgentMiddleware[AgentState[ResponseT], ContextT, ResponseT]`

## Constructor

```python
SummarizationMiddleware(
    model: str | BaseChatModel, # Model used to generate summaries
    *,
    trigger: (
        ContextSize
        | TriggerClause
        | list[ContextSize | TriggerClause]
        | None
    ) = None, # Conditions that trigger summarization
    keep: ContextSize = ("messages", 20), # Recent context to preserve
    token_counter: TokenCounter = count_tokens_approximately,
    summary_prompt: str = DEFAULT_SUMMARY_PROMPT,
    trim_tokens_to_summarize: int | None = 4000,
    **deprecated_kwargs: Any
)
```

## Parameters

* `model` — Language model used to generate the replacement summary.
  * May be a model identifier string.
  * May be an initialized `BaseChatModel`.
  * String identifiers are initialized internally using `init_chat_model`.

* `trigger` — Threshold or thresholds that determine when summarization runs.
  * Default: `None`
  * When `None`, automatic summarization is disabled.
  * May be a single context-size tuple.
  * May be one dictionary containing multiple thresholds.
  * May be a list mixing tuples and dictionaries.

* `keep` — Determines how much recent history remains after summarization.
  * Default: `("messages", 20)`
  * Supports message count, token count, or a fraction of the model's maximum input tokens.
  * Accepts only one `ContextSize` value.

* `token_counter` — Function used to count tokens in message collections.
  * Default: `count_tokens_approximately`
  * When the default counter is used, the middleware enables usage-metadata scaling.
  * Anthropic chat models use an estimated `3.3` characters per token.

* `summary_prompt` — Prompt template sent to the summarization model.
  * Default: `DEFAULT_SUMMARY_PROMPT`
  * Must contain the `{messages}` placeholder.
  * The default prompt's `<messages>` marker is part of its public contract.

* `trim_tokens_to_summarize` — Maximum number of message tokens passed to the summary-generation call.
  * Default: `4000`
  * Set to `None` to disable trimming.

* `**deprecated_kwargs` — Supports deprecated constructor arguments for compatibility.
  * `max_tokens_before_summary`
  * `messages_to_keep`

## Deprecated Parameters

### `max_tokens_before_summary`

```python
SummarizationMiddleware(
    model,
    max_tokens_before_summary=4000
)
```

This emits a `DeprecationWarning` and is equivalent to:

```python
SummarizationMiddleware(
    model,
    trigger=("tokens", 4000)
)
```

The deprecated value is used only when `trigger` was not explicitly supplied.

### `messages_to_keep`

```python
SummarizationMiddleware(
    model,
    messages_to_keep=10
)
```

This emits a `DeprecationWarning` and is equivalent to:

```python
SummarizationMiddleware(
    model,
    keep=("messages", 10)
)
```

The deprecated value replaces `keep` only when `keep` still has its default value.

## Attributes

* `model` — Initialized model used to create summaries.
* `trigger` — Copied version of the user-provided trigger configuration.
* `keep` — Validated context-retention configuration.
* `token_counter` — Token-counting function used for complete message collections.
* `summary_prompt` — Prompt template used for summary generation.
* `trim_tokens_to_summarize` — Token budget for the summarization request.
* `_trigger_clauses` — Canonical list of trigger clauses.
  * Conditions inside one clause use AND semantics.
  * Separate clauses use OR semantics.
* `_trigger_conditions` — Legacy private tuple-based compatibility representation.
* `_partial_token_counter` — Counter used while searching for a token-based cutoff.
  * Usage-metadata scaling is disabled for partial message slices.

# Type Aliases

## `TokenCounter`

Callable used to count the tokens in a collection of message representations.

```python
TokenCounter = Callable[
    [Iterable[MessageLikeRepresentation]],
    int
]
```

## `ContextFraction`

Specifies a fraction of the summarization model's maximum input-token capacity.

```python
ContextFraction = tuple[
    Literal["fraction"],
    float
]
```

Example:

```python
("fraction", 0.8)
```

A fractional value must be greater than `0` and less than or equal to `1`.

## `ContextTokens`

Specifies an absolute token count.

```python
ContextTokens = tuple[
    Literal["tokens"],
    int
]
```

Example:

```python
("tokens", 3000)
```

The token value must be greater than `0`.

## `ContextMessages`

Specifies an absolute message count.

```python
ContextMessages = tuple[
    Literal["messages"],
    int
]
```

Example:

```python
("messages", 50)
```

The message value must be greater than `0`.

## `ContextSize`

Union of the three supported context-size formats.

```python
ContextSize = (
    ContextFraction
    | ContextTokens
    | ContextMessages
)
```

The meaning depends on where it is used:

* In `trigger`, it specifies when summarization begins.
* In `keep`, it specifies how much recent context remains afterward.

# `TriggerClause`

Dictionary-based trigger specification.

```python
class TriggerClause(TypedDict, total=False):
    tokens: int
    messages: int
    fraction: float
```

## Fields

* `tokens` — Trigger when the computed or provider-reported token count reaches this value.
* `messages` — Trigger when the number of messages reaches this value.
* `fraction` — Trigger when tokens reach this fraction of the model's maximum input limit.

All fields are optional at the type level, but a runtime clause must contain at least one supported field.

## Trigger Semantics

Conditions inside a dictionary use **AND** logic.

```python
{
    "tokens": 4000,
    "messages": 10
}
```

This triggers only when:

```text
token count >= 4000
AND
message count >= 10
```

Items in a list use **OR** logic.

```python
[
    {"tokens": 5000, "messages": 3},
    {"tokens": 3000, "messages": 6},
]
```

This triggers when:

```text
(tokens >= 5000 AND messages >= 3)
OR
(tokens >= 3000 AND messages >= 6)
```

A tuple in the list represents a one-condition clause:

```python
[
    ("fraction", 0.8),
    ("messages", 100),
]
```

This triggers when either threshold is met.

# `DEFAULT_SUMMARY_PROMPT`

Default prompt used to extract the most important context from older conversation history.

The requested summary is structured into:

```text
## SESSION INTENT
## SUMMARY
## ARTIFACTS
## NEXT STEPS
```

The prompt requires the model to return only the extracted context.

These elements are part of the constant's public contract:

```text
<messages>
```

and:

```python
{messages}
```

Changing, removing, or reformatting them may break downstream consumers that modify the prompt through string replacement.

# Methods

## 1. `before_model`

Processes conversation history before a synchronous model invocation.

```python
before_model(
    self,
    state: AgentState[Any],
    runtime: Runtime[ContextT]
) -> dict[str, Any] | None
```

### Behaviour

1. Reads `state["messages"]`.
2. Assigns UUIDs to messages that have no ID.
3. Calculates the current token count.
4. Checks all configured trigger clauses.
5. Determines a safe cutoff index.
6. Summarizes messages before the cutoff.
7. Preserves messages from the cutoff onward.
8. Returns a message-state replacement.

The returned update has this structure:

```python
{
    "messages": [
        RemoveMessage(id=REMOVE_ALL_MESSAGES),
        HumanMessage(
            content=(
                "Here is a summary of the conversation "
                "to date:\n\n<generated summary>"
            ),
            additional_kwargs={
                "lc_source": "summarization"
            },
        ),
        *preserved_messages,
    ]
}
```

Returns `None` when:

* No trigger is configured.
* No trigger clause is satisfied.
* The retention policy produces no safe messages to summarize.

## 2. `abefore_model`

Asynchronous version of `before_model`.

```python
async def abefore_model(
    self,
    state: AgentState[Any],
    runtime: Runtime[ContextT]
) -> dict[str, Any] | None
```

It uses `model.ainvoke` for summary generation.

## 3. `_copy_trigger`

Copies mutable trigger containers.

```python
_copy_trigger(
    trigger: (
        ContextSize
        | TriggerClause
        | list[ContextSize | TriggerClause]
        | None
    )
) -> (
    ContextSize
    | TriggerClause
    | list[ContextSize | TriggerClause]
    | None
)
```

This prevents later mutations to the caller's dictionaries or list from changing the middleware configuration.

## 4. `_legacy_trigger_conditions`

Projects tuple-compatible trigger values into the earlier private representation.

```python
_legacy_trigger_conditions(
    self,
    trigger: (
        ContextSize
        | TriggerClause
        | list[ContextSize | TriggerClause]
        | None
    )
) -> list[ContextSize]
```

Multi-condition dictionaries cannot be represented in this legacy list and are omitted.

The canonical behaviour is controlled by `_trigger_clauses`.

## 5. `_normalize_trigger`

Validates and normalizes trigger input into a list of `TriggerClause` dictionaries.

```python
_normalize_trigger(
    self,
    trigger: (
        ContextSize
        | TriggerClause
        | list[ContextSize | TriggerClause]
        | None
    )
) -> list[TriggerClause]
```

Examples:

```text
("tokens", 3000)
    -> [{"tokens": 3000}]

{"tokens": 4000, "messages": 10}
    -> [{"tokens": 4000, "messages": 10}]

[("messages", 50), {"tokens": 4000, "messages": 10}]
    -> [{"messages": 50}, {"tokens": 4000, "messages": 10}]
```

It rejects:

* Empty trigger dictionaries.
* Unsupported metric names.
* Boolean thresholds.
* Non-integer token or message thresholds.
* Non-numeric fraction thresholds.
* Unsupported trigger container types.
* Unsupported items inside a trigger list.

## 6. `_should_summarize_based_on_reported_tokens`

Checks token usage reported by the latest `AIMessage`.

```python
_should_summarize_based_on_reported_tokens(
    self,
    messages: list[AnyMessage],
    threshold: float
) -> bool
```

Reported tokens are used only when:

* The latest AI message has `usage_metadata`.
* `usage_metadata["total_tokens"]` reaches the threshold.
* The message contains `response_metadata["model_provider"]`.
* The reported provider matches the summary model's LangSmith provider.

Known provider aliases include:

```python
{
    "amazon_bedrock": {
        "bedrock",
        "bedrock_converse",
    }
}
```

## 7. `_should_summarize`

Evaluates the normalized trigger clauses.

```python
_should_summarize(
    self,
    messages: list[AnyMessage],
    total_tokens: int
) -> bool
```

Rules:

* No configured clauses means `False`.
* All metrics within one clause must be satisfied.
* Any satisfied clause returns `True`.
* Token thresholds may use either calculated tokens or matching provider-reported tokens.
* Fraction thresholds are converted to absolute tokens using the model profile.

## 8. `_determine_cutoff_index`

Chooses the index separating summarized and preserved messages.

```python
_determine_cutoff_index(
    self,
    messages: list[AnyMessage]
) -> int
```

For message-based retention, it preserves the configured number of recent messages.

For token- or fraction-based retention, it searches for a suffix within the target token budget.

## 9. `_find_token_based_cutoff`

Finds the earliest safe message index whose remaining suffix fits within the retention token limit.

```python
_find_token_based_cutoff(
    self,
    messages: list[AnyMessage]
) -> int | None
```

The method uses binary search.

For partial suffixes, the middleware does not use usage-metadata scaling because provider-reported totals apply to complete requests, not arbitrary message slices.

Returns:

* `0` when no summarization is required.
* A positive cutoff index when older messages should be summarized.
* `None` when a fractional limit cannot obtain model-profile information.

## 10. `_get_profile_limits`

Retrieves the model's maximum input-token limit.

```python
_get_profile_limits(
    self
) -> int | None
```

It reads:

```python
model.profile["max_input_tokens"]
```

Returns `None` when:

* The model has no `profile` attribute.
* The profile is not a mapping.
* `max_input_tokens` is not an integer.

## 11. `_validate_context_size`

Validates one `ContextSize` tuple.

```python
_validate_context_size(
    context: ContextSize,
    parameter_name: str
) -> ContextSize
```

Validation rules:

```text
fraction: 0 < value <= 1
tokens:   value > 0
messages: value > 0
```

Unsupported context-size kinds raise `ValueError`.

## 12. `_build_new_messages`

Creates the replacement summary message.

```python
_build_new_messages(
    summary: str
) -> list[HumanMessage]
```

The generated message begins with:

```text
Here is a summary of the conversation to date:
```

It also includes:

```python
additional_kwargs={
    "lc_source": "summarization"
}
```

## 13. `_ensure_message_ids`

Assigns UUID strings to messages missing IDs.

```python
_ensure_message_ids(
    messages: list[AnyMessage]
) -> None
```

Message IDs are required so LangGraph's `add_messages` reducer can correctly process removals and replacements.

## 14. `_partition_messages`

Splits messages at the selected cutoff.

```python
_partition_messages(
    conversation_messages: list[AnyMessage],
    cutoff_index: int
) -> tuple[
    list[AnyMessage],
    list[AnyMessage]
]
```

Returns:

```python
(
    conversation_messages[:cutoff_index],
    conversation_messages[cutoff_index:],
)
```

## 15. `_find_safe_cutoff`

Finds a message-count-based cutoff that preserves tool-call relationships.

```python
_find_safe_cutoff(
    self,
    messages: list[AnyMessage],
    messages_to_keep: int
) -> int
```

Returns `0` when the total message count is already within the retention limit.

## 16. `_find_safe_cutoff_point`

Adjusts a cutoff so an AI tool-call message and its tool results remain together.

```python
_find_safe_cutoff_point(
    messages: list[AnyMessage],
    cutoff_index: int
) -> int
```

When the proposed cutoff points to a `ToolMessage`, the method:

1. Collects tool-call IDs from consecutive tool messages.
2. Searches backward for the `AIMessage` containing matching tool calls.
3. Moves the cutoff to that AI message.

When no matching AI message is found, it advances beyond the consecutive tool messages so orphaned tool responses are not preserved.

## 17. `_create_summary`

Synchronously generates a summary.

```python
_create_summary(
    self,
    messages_to_summarize: list[AnyMessage]
) -> str
```

Behaviour:

* Empty input returns:
  ```text
  No previous conversation history.
  ```
* Messages are trimmed before summarization.
* Trimmed messages are serialized in XML format.
* The model is called with:
  ```python
  config={
      "metadata": {
          "lc_source": "summarization"
      }
  }
  ```
* The returned model text is stripped.
* Model exceptions are converted to:
  ```text
  Error generating summary: <exception>
  ```

The exception is not re-raised.

## 18. `_acreate_summary`

Asynchronous version of `_create_summary`.

```python
async def _acreate_summary(
    self,
    messages_to_summarize: list[AnyMessage]
) -> str
```

It uses `model.ainvoke`.

## 19. `_trim_messages_for_summary`

Trims messages before passing them to the summarization model.

```python
_trim_messages_for_summary(
    self,
    messages: list[AnyMessage]
) -> list[AnyMessage]
```

When trimming is enabled, it calls `trim_messages` using:

```python
trim_messages(
    messages,
    max_tokens=trim_tokens_to_summarize,
    token_counter=token_counter,
    start_on="human",
    strategy="last",
    allow_partial=True,
    include_system=True,
)
```

When `trim_tokens_to_summarize=None`, it returns all messages unchanged.

If trimming raises an exception, it falls back to the last `15` messages.

If trimming returns no messages, summary generation returns:

```text
Previous conversation was too long to summarize.
```

# Fractional Limits

Fractional `trigger` or `keep` values require model profile information.

Example:

```python
SummarizationMiddleware(
    model,
    trigger=("fraction", 0.8),
    keep=("fraction", 0.3),
)
```

The middleware requires:

```python
model.profile = {
    "max_input_tokens": <integer>
}
```

When profile information is unavailable, constructor initialization raises `ValueError`.

Example calculation:

```text
max_input_tokens = 100000
trigger = ("fraction", 0.8)
keep = ("fraction", 0.3)

Trigger threshold = 80000 tokens
Retention target = 30000 tokens
```

# Summarization Flow

```text
before_model / abefore_model
        |
        v
Ensure every message has an ID
        |
        v
Count current tokens
        |
        v
Does any trigger clause match?
        |
   No --+--> Return None
        |
       Yes
        |
        v
Determine safe cutoff from keep policy
        |
        v
Is cutoff greater than zero?
        |
   No --+--> Return None
        |
       Yes
        |
        v
Summarize messages before cutoff
        |
        v
Remove all existing messages
        |
        v
Insert summary HumanMessage
        |
        v
Append preserved recent messages
```

# Examples

## Trigger by Message Count

```python
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware

agent = create_agent(
    model="openai:gpt-5.5",
    middleware=[
        SummarizationMiddleware(
            model="openai:gpt-5.5",
            trigger=("messages", 50),
            keep=("messages", 20),
        )
    ],
)
```

Summarization begins when the history reaches `50` messages and preserves approximately the most recent `20`, adjusted when necessary to keep tool-call pairs together.

## Trigger by Token Count

```python
middleware = SummarizationMiddleware(
    model="openai:gpt-5.5",
    trigger=("tokens", 8000),
    keep=("tokens", 3000),
)
```

## Trigger by Fraction

```python
middleware = SummarizationMiddleware(
    model=chat_model,
    trigger=("fraction", 0.8),
    keep=("fraction", 0.3),
)
```

The model must expose a valid `profile["max_input_tokens"]`.

## AND Trigger

```python
middleware = SummarizationMiddleware(
    model="openai:gpt-5.5",
    trigger={
        "tokens": 4000,
        "messages": 10,
    },
)
```

Both conditions must be true.

## OR Triggers

```python
middleware = SummarizationMiddleware(
    model="openai:gpt-5.5",
    trigger=[
        ("tokens", 8000),
        ("messages", 100),
    ],
)
```

Either threshold can trigger summarization.

## Combined AND/OR Triggers

```python
middleware = SummarizationMiddleware(
    model="openai:gpt-5.5",
    trigger=[
        {
            "tokens": 5000,
            "messages": 3,
        },
        {
            "tokens": 3000,
            "messages": 6,
        },
    ],
)
```

## Custom Token Counter

```python
def count_tokens(messages) -> int:
    return sum(
        len(str(message.content).split())
        for message in messages
    )

middleware = SummarizationMiddleware(
    model="openai:gpt-5.5",
    trigger=("tokens", 5000),
    token_counter=count_tokens,
)
```

## Disable Summary-Input Trimming

```python
middleware = SummarizationMiddleware(
    model="openai:gpt-5.5",
    trigger=("messages", 50),
    trim_tokens_to_summarize=None,
)
```

## Custom Summary Prompt

```python
custom_prompt = """
Summarize the important facts, completed work, files, and next steps.

Conversation:
{messages}
"""

middleware = SummarizationMiddleware(
    model="openai:gpt-5.5",
    trigger=("messages", 50),
    summary_prompt=custom_prompt,
)
```

# Exceptions

The constructor can raise `ValueError` when:

* A fractional trigger or retention policy is used without model-profile limits.
* A fraction is not between `0` and `1`.
* A token or message threshold is not greater than `0`.
* A context-size kind is unsupported.
* A trigger dictionary is empty.
* A trigger dictionary contains an unsupported metric.
* A trigger threshold has an invalid value type.

The constructor can raise `TypeError` when:

* `trigger` has an unsupported container type.
* A trigger list contains an item that is neither a tuple nor a mapping.

Errors from the summarization model are caught and converted to summary text instead of being raised.

# Source

This reference follows the pinned LangChain source:

```text
libs/langchain_v1/langchain/agents/middleware/summarization.py
Commit: 42f8f79293cfb7589e5bc1d74a8ae4dfd0bf15e3
```